# **Encounters Data Cleaning**

## 1. Importing Libraries and Loading Data

In [3]:
import pandas as pd
import numpy as np

from pathlib import Path

import sys
sys.path.append(str(Path.cwd().parent))

from scripts.paths import RAW_DIR, CLEANED_DIR

df = pd.read_csv(RAW_DIR/"05_encounters_dirty.csv")

## 2. Initial Data Inspection

In [4]:
df.head()

,Encounter_ID,Patient_ID,Department_ID,Doctor_ID,Encounter_Date,Visit_Type,Admission_Date,Discharge_Date,Length_of_Stay,Outcome,Room_Bed_ID
0,E200000,P100977,D012,S10233,2026-02-01,opd,NaN,NaN,0.0,Discharged,NaN
1,E200001,P105027,D003,S10270,2025-05-17,IPD,2025-05-17 20:00:00,2025-05-27 22:00:00,10.0,Expired,NaN
2,E200002,P104136,D014,S10099,2026-02-10,OPD,NaN,NaN,0.0,Transferred,B1371
3,E200003,P106901,D011,S10363,2025-06-23,IPD,2025-06-23 10:00:00,2025-07-03 12:00:00,10.0,LAMA,NaN
4,E200004,P103900,D010,S10523,2025-05-14,Emergency,2025-05-14 06:00:00,2025-05-26 06:00:00,12.0,NaN,B1642


In [5]:
df.shape

(15000, 11)

In [6]:
df.columns.tolist()

['Encounter_ID',
 'Patient_ID',
 'Department_ID',
 'Doctor_ID',
 'Encounter_Date',
 'Visit_Type',
 'Admission_Date',
 'Discharge_Date',
 'Length_of_Stay',
 'Outcome',
 'Room_Bed_ID']

In [8]:
df.dtypes

Encounter_ID       object
Patient_ID         object
Department_ID      object
Doctor_ID          object
Encounter_Date     object
Visit_Type         object
Admission_Date     object
Discharge_Date     object
Length_of_Stay    float64
Outcome            object
Room_Bed_ID        object
dtype: object

In [9]:
df.isna().sum()

Encounter_ID          0
Patient_ID            0
Department_ID        62
Doctor_ID             0
Encounter_Date        0
Visit_Type         2130
Admission_Date    11612
Discharge_Date    11612
Length_of_Stay       54
Outcome            2574
Room_Bed_ID       10022
dtype: int64

In [10]:
df.duplicated().sum()

0

## 3. Audit of Specific Columns

In [11]:
df["Encounter_ID"].duplicated().sum()

0

In [14]:
df["Patient_ID"].duplicated().sum()

8208

In [13]:
df["Doctor_ID"].duplicated().sum()

14100

In [12]:
df["Room_Bed_ID"].duplicated().sum()

14299

In [15]:
df["Visit_Type"].value_counts(dropna=False)

Visit_Type
Follow-up    2186
OPD          2182
Emergency    2173
NaN          2130
IPD          2124
opd          2122
OPD          2083
Name: count, dtype: int64

In [16]:
df["Outcome"].value_counts(dropna=False)

Outcome
NaN            2574
Discharged     2540
Expired        2515
Transferred    2463
LAMA           2463
Recovered      2445
Name: count, dtype: int64

In [17]:
df["Department_ID"].value_counts(dropna=False)

Department_ID
D009    873
D017    861
D018    857
D013    848
D004    848
D002    838
D007    832
D015    829
D016    827
D001    821
D003    821
D012    820
D014    814
D010    807
D008    805
D011    802
D006    794
D005    779
NaN      62
D01      35
d005     27
Name: count, dtype: int64

In [18]:
department_check = (
    df["Department_ID"]
    .astype("string")
    .str.strip()
    .str.upper()
)

department_check.value_counts(dropna=False)

Department_ID
D009    873
D017    861
D018    857
D013    848
D004    848
D002    838
D007    832
D015    829
D016    827
D001    821
D003    821
D012    820
D014    814
D010    807
D005    806
D008    805
D011    802
D006    794
<NA>     62
D01      35
Name: count, dtype: Int64

In [25]:
df["Encounter_Date"].dtype

dtype('O')

In [21]:
df["Admission_Date"].dtype

dtype('O')

In [22]:
df["Discharge_Date"].dtype

dtype('O')

In [23]:
encounter_temp = pd.to_datetime(
    df["Encounter_Date"],
    errors="coerce"
)

admission_temp = pd.to_datetime(
    df["Admission_Date"],
    errors="coerce"
)

discharge_temp = pd.to_datetime(
    df["Discharge_Date"],
    errors="coerce"
)

In [26]:
encounter_temp.isna().sum()

0

In [27]:
admission_temp.isna().sum()

11612

In [28]:
discharge_temp.isna().sum()

11612

In [29]:
(admission_temp > discharge_temp).sum()

0

In [30]:
(admission_temp < encounter_temp).sum()

0

In [31]:
df["Length_of_Stay"].describe()

count    14946.000000
mean         2.812191
std          7.742095
min         -2.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         90.000000
Name: Length_of_Stay, dtype: float64

In [32]:
(df["Length_of_Stay"] < 0).sum()

80

In [33]:
df["Length_of_Stay"].value_counts().sort_index()

Length_of_Stay
-2.0        39
-1.0        41
 0.0     11423
 1.0       183
 2.0       176
 3.0       163
 4.0       160
 5.0       194
 6.0       158
 7.0       152
 8.0       169
 9.0       154
 10.0      165
 11.0      190
 12.0      169
 13.0      161
 14.0      150
 15.0      148
 16.0      147
 17.0      155
 18.0      176
 19.0      180
 20.0      177
 45.0       66
 90.0       50
Name: count, dtype: int64

In [34]:
stay_expected = (
    discharge_temp - admission_temp
).dt.days

In [35]:
stay_mask = admission_temp.notna() & discharge_temp.notna()

(
    df.loc[stay_mask, "Length_of_Stay"]
    != stay_expected[stay_mask]
).sum()

61

In [36]:
departments_df = pd.read_csv(
    CLEANED_DIR / "02_departments_cleaned.csv"
)

valid_department_ids = departments_df["Department_ID"].dropna().unique()

df.loc[
    ~department_check.isin(valid_department_ids),
    "Department_ID"
].value_counts(dropna=False)

Department_ID
NaN    62
D01    35
Name: count, dtype: int64

In [37]:
beds_df = pd.read_csv(
    CLEANED_DIR / "04_beds_cleaned.csv"
)

valid_bed_ids = beds_df["Bed_ID"].dropna().unique()

df.loc[
    df["Room_Bed_ID"].notna() &
    ~df["Room_Bed_ID"].isin(valid_bed_ids),
    "Room_Bed_ID"
]

Series([], Name: Room_Bed_ID, dtype: object)

## 4. Data Cleaning

### 4.1 Department ID

In [38]:
department_cleaned = (
    df["Department_ID"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace({
        "D01": "D001"
    })
)

In [39]:
department_cleaned.value_counts(dropna=False)

Department_ID
D009    873
D017    861
D018    857
D001    856
D013    848
D004    848
D002    838
D007    832
D015    829
D016    827
D003    821
D012    820
D014    814
D010    807
D005    806
D008    805
D011    802
D006    794
<NA>     62
Name: count, dtype: Int64

In [40]:
df["Department_ID"] = department_cleaned

### 4.2 Visit Type

In [41]:
visit_cleaned = (
    df["Visit_Type"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "opd": "OPD",
        "ipd": "IPD",
        "emergency": "Emergency",
        "follow-up": "Follow-up"
    })
)

In [42]:
visit_cleaned.value_counts(dropna=False)

Visit_Type
OPD          6387
Follow-up    2186
Emergency    2173
<NA>         2130
IPD          2124
Name: count, dtype: Int64

In [43]:
df["Visit_Type"] = visit_cleaned

### 4.3 Clean Dates

In [44]:
encounter_cleaned = pd.to_datetime(
    df["Encounter_Date"],
    errors="coerce"
)

admission_cleaned = pd.to_datetime(
    df["Admission_Date"],
    errors="coerce"
)

discharge_cleaned = pd.to_datetime(
    df["Discharge_Date"],
    errors="coerce"
)

In [45]:
df["Encounter_Date"] = encounter_cleaned
df["Admission_Date"] = admission_cleaned
df["Discharge_Date"] = discharge_cleaned

### 4.4 Length of Stay

In [46]:
length_cleaned = pd.Series(
    0,
    index=df.index,
    dtype="Int64"
)

stay_mask = (
    df["Admission_Date"].notna() &
    df["Discharge_Date"].notna()
)

length_cleaned.loc[stay_mask] = (
    df.loc[stay_mask, "Discharge_Date"]
    - df.loc[stay_mask, "Admission_Date"]
).dt.days.astype("Int64")

In [47]:
df["Length_of_Stay"] = length_cleaned

## 5. Final Validation

In [48]:
df.shape

(15000, 11)

In [49]:
df.dtypes

Encounter_ID              object
Patient_ID                object
Department_ID     string[python]
Doctor_ID                 object
Encounter_Date    datetime64[ns]
Visit_Type        string[python]
Admission_Date    datetime64[ns]
Discharge_Date    datetime64[ns]
Length_of_Stay             Int64
Outcome                   object
Room_Bed_ID               object
dtype: object

In [50]:
df.isna().sum()

Encounter_ID          0
Patient_ID            0
Department_ID        62
Doctor_ID             0
Encounter_Date        0
Visit_Type         2130
Admission_Date    11612
Discharge_Date    11612
Length_of_Stay        0
Outcome            2574
Room_Bed_ID       10022
dtype: int64

In [51]:
df.duplicated().sum()

0

In [52]:
df["Encounter_ID"].duplicated().sum()

0

In [53]:
df["Department_ID"].value_counts(dropna=False)

Department_ID
D009    873
D017    861
D018    857
D001    856
D013    848
D004    848
D002    838
D007    832
D015    829
D016    827
D003    821
D012    820
D014    814
D010    807
D005    806
D008    805
D011    802
D006    794
<NA>     62
Name: count, dtype: Int64

In [54]:
df["Visit_Type"].value_counts(dropna=False)

Visit_Type
OPD          6387
Follow-up    2186
Emergency    2173
<NA>         2130
IPD          2124
Name: count, dtype: Int64

In [55]:
df["Outcome"].value_counts(dropna=False)

Outcome
NaN            2574
Discharged     2540
Expired        2515
Transferred    2463
LAMA           2463
Recovered      2445
Name: count, dtype: int64

In [56]:
(df["Admission_Date"] > df["Discharge_Date"]).sum()

0

In [57]:
(df["Admission_Date"] < df["Encounter_Date"]).sum()

0

In [58]:
df["Length_of_Stay"].isna().sum()

0

In [59]:
(df["Length_of_Stay"] < 0).sum()

0

In [60]:
df.loc[
    df["Department_ID"].notna() &
    ~df["Department_ID"].isin(valid_department_ids),
    "Department_ID"
]

Series([], Name: Department_ID, dtype: string)

## 6. Export Cleaned Data

In [61]:
df.to_csv(CLEANED_DIR / "05_encounters_cleaned.csv", index=False)